Physics Informed DeepONet on a integrator problem. 

In [1]:
import qewton

path = "/localdata/tomfre/SolutionData/Integrator/"

F = qewton.config.Variable("f", 1)
X = qewton.config.Variable("x", 1)
U = qewton.config.Variable("u", 1)

In [18]:
f_data = qewton.DEFAULT_DL_BACKEND.load_data(path + "f_data.pt")
u_data = qewton.DEFAULT_DL_BACKEND.load_data(path + "u_data.pt")
x_data = qewton.DEFAULT_DL_BACKEND.load_data(path + "x_data.pt")
initial_x_data = qewton.DEFAULT_DL_BACKEND.load_data(path + "initial_x.pt")

train_f_data, train_x_data, train_initial_data = f_data[:4000], x_data[:4000], initial_x_data[:4000]
test_f_data, test_u_data, test_x_data = f_data[4000:], u_data[4000:], x_data[4000:]

batch_size = 1000

In [21]:
x_data.shape

torch.Size([5000, 100, 1])

In [20]:
f_data.shape

torch.Size([5000, 100, 1])

In [ ]:
dataset = qewton.data.GridDataSet(
    data=[train_f_data, train_x_data], 
    feature_variables=[F, X],
    geometry_variable=X,
    point_grid=x_data[0]
)

dataset = qewton.data.GridDataSet(
    data=[train_f_data, train_x_data], 
    feature_variables=[F, X],
    geometry_variable=X,
    point_grid=x_data[0]
)

In [9]:
data_loader = qewton.data.DataLoader(
    data_set=dataset,
    batch_size=batch_size,
    splitting_ratio=(1.0, 0.0, 0.0),
    shuffle_data=False,
)

In [ ]:
model = qewton.algorithms.FCNDeepONet(
    trunk_input=X, 
    branch_input=F,
    output=U,
    trunk_hidden_neurons=16,
    branch_hidden_neurons=16,
    trunk_hidden_layers=3,
    branch_hidden_layers=2,
    intermediate_neurons=32,
    activations=qewton.bb.Tanh
)

In [ ]:
def ode_residual(x: X, f: F, u: U):
    return u.gradient(x) - f

ode_constraint = qewton.constraints.MSEConstraint()
ode_pipeline = qewton.Graph()

with ode_pipeline.tracker(0):
    f, x = data_loader()
    model_out = model(f, x)
    ode_res = ode_residual(x, f, model_out)
    ode_constraint(ode_res, 0.0)

In [ ]:
def initial_residual(u: U):
    return u 

initial_constraint = qewton.constraints.MSEConstraint()
initial_pipeline = qewton.Graph()

with initial_pipeline.tracker(0):
    f, x = data_loader()
    model_out = model(f, x)
    initial_res = initial_residual(model_out)
    initial_constraint(initial_res, 0.0)

In [17]:
adam_phase = qewton.optim.OptimizationPhase(
    optimizer=qewton.optim.Adam(),
    lr=0.001,
    max_iterations=2500,
)

trainer = qewton.optim.GraphBasedTrainer(
    optimization_phases=[adam_phase],
    graphs=[computation_graph],
    training_objectives=[constraint],
    device="cuda:0",
)

trainer.run()

Optimization Phase 1: 100%|██████████| 2500/2500 [01:12<00:00, 34.32it/s, loss=0.00043] 


In [ ]:
test_dataset = qewton.data.GridDataSet(
    data=[test_f_data, test_x_data, test_u_data],
    feature_variables=[F, X, U],
    geometry_variable=X,
    point_grid=x_data[0]
)


u_config = data_loader.get_output_port(U).get_data_configuration(computation_graph)
layout = computation_graph.visualize(
    model.output_ports[0],
    reference=data_loader.get_output_port(U),
    prediction_config=u_config,
    mode=qewton.optim.EvaluationPhase.TEST,
)
qewton.visualization.Figure(layout).show()